# Interactive Optimization with ADM and Pymoo-based Problem Server

This notebook demonstrates how to use an ADM framework to interactively solve benchmark problems (DTLZ, WFG) exposed through a FastAPI problem server.

## 1. Introduction

This notebook demonstrates how to connect to a FastAPI-based *Problem Server* that exposes
multi-objective benchmark problems (e.g., DTLZ and WFG families) through a REST API.

⚠️ **Important:** Before running this notebook, make sure the problem server (` desdeo\problem\testproblems\benchmark_server.py`) is running locally.

You can start it by opening a terminal and running:

```bash
python benchmark_server.py
```

The server should be accessible at `http://127.0.0.1:8000`.

Once the server is up, this notebook will:

1. Connect to it to create problem instances via `server_problem()`.
2. Dynamically generate DTLZ and WFG problems with customizable numbers of objectives and variables.
3. Solve them using the Artificial Decision Maker (ADM) framework and interactive evolutionary algorithms (NSGA-III and RVEA).
4. Visualize the ADM’s learning and decision phases.

## 2. Imports and setup


In [1]:
import numpy as np
import pandas as pd
from desdeo.problem import Problem
from desdeo.emo.methods.EAs import ReferenceVectorOptions, nsga3, rvea
from desdeo.adm.ADMAfsar import ADMAfsar
from desdeo.emo.hooks.archivers import NonDominatedArchive
from desdeo.problem.testproblems.benchmarks_server import PymooParameters, server_problem
import time
import os

# -- Configuration: Server URL and port
SERVER_URL = "http://127.0.0.1"
SERVER_PORT = 8000

OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Ready to connect to Problem Server at:", f"{SERVER_URL}:{SERVER_PORT}")

✅ Ready to connect to Problem Server at: http://127.0.0.1:8000


## 3. Define Helper Functions for DTLZ and WFG Problems

DTLZ and WFG problems have different default numbers of variables depending on the number of objectives.
These helper functions encapsulate that logic.

In [2]:
def get_dtlz_default_nvar(problem_name: str, n_obj: int) -> int:
    """
    Get the recommended number of variables for a given DTLZ problem and number of objectives.
    Source: pymoo defaults.
    """
    base = {
        "dtlz1": 5,
        "dtlz2": 10,
        "dtlz3": 10,
        "dtlz4": 10,
        "dtlz5": 10,
        "dtlz6": 10,
        "dtlz7": 20,
    }
    k = base.get(problem_name, 10)
    return n_obj + k - 1


def get_wfg_default_nvar(problem_name: str, n_obj: int) -> int:
    """
    WFG problems have n_var = k + l where typically:
      k = 2 * (n_obj - 1)
      l = 20
    """
    k = 2 * (n_obj - 1)
    l = 20
    return k + l

## 4. Problem Creation via the Server


In [3]:
def get_default_ea_parameters(problem_name: str, n_obj: int):
    """
    Returns the default population size, number of generations, and multi-layer
    reference vector configurations for NSGA-III / RVEA depending on the problem
    type and number of objectives.

    The population size is automatically calculated based on the number of reference
    vectors that will be generated from the multi-layer configuration.

    Parameters
    ----------
    problem_name : str
        Name of the problem (e.g. "dtlz1", "wfg2", etc.)
    n_obj : int
        Number of objectives.

    Returns
    -------
    pop_size : int
        Suggested population size (based on number of reference vectors).
        Always an even number.
    n_gen : int
        Suggested number of generations.
    pymoo_layers : list[dict]
        Default multi-layer configuration for reference vectors.
        Each dict contains 'strategy', 'n_partitions', and 'scaling' keys.
    """
    from scipy.special import comb
    
    problem_name = problem_name.lower()

    # Default generation counts
    dtlz_gen_defaults = {
        3:  250,
        5:  350,
        7:  400,
        9:  500
    }

    wfg_gen_defaults = {
        3:  400,
        5:  500,
        7:  600,
        9:  800
    }

    # Default multi-layer configurations based on number of objectives
    # Two-layer approach with different scaling factors
    if n_obj == 3:
        # For 3 objectives, use standard das-dennis with two scales
        pymoo_layers = [
            {'strategy': 'das-dennis', 'n_partitions': 12, 'scaling': 1.0},
            {'strategy': 'das-dennis', 'n_partitions': 12, 'scaling': 0.5},
        ]
    elif n_obj == 5:
        # For 5 objectives, use slightly coarser partitions
        pymoo_layers = [
            {'strategy': 'das-dennis', 'n_partitions': 6, 'scaling': 1.0},
            {'strategy': 'das-dennis', 'n_partitions': 6, 'scaling': 0.5},
        ]
    elif n_obj == 7:
        # For 7 objectives, use even coarser partitions
        pymoo_layers = [
            {'strategy': 'das-dennis', 'n_partitions': 4, 'scaling': 1.0},
            {'strategy': 'das-dennis', 'n_partitions': 4, 'scaling': 0.5},
        ]
    elif n_obj >= 9:
        # For 9+ objectives, use minimal partitions
        pymoo_layers = [
            {'strategy': 'das-dennis', 'n_partitions': 3, 'scaling': 1.0},
            {'strategy': 'das-dennis', 'n_partitions': 3, 'scaling': 0.5},
        ]
    else:
        # Fallback for other objective counts
        pymoo_layers = [
            {'strategy': 'das-dennis', 'n_partitions': 12, 'scaling': 1.0},
            {'strategy': 'das-dennis', 'n_partitions': 12, 'scaling': 0.5},
        ]

    # Calculate the number of reference vectors from the layers
    # For das-dennis, the number of vectors per layer is: C(H + M - 1, M - 1)
    # where H is n_partitions and M is n_obj
    total_vectors = 0
    for layer in pymoo_layers:
        if layer['strategy'] == 'das-dennis':
            H = layer['n_partitions']
            # Number of reference vectors in this layer
            n_vectors = int(comb(H + n_obj - 1, n_obj - 1, exact=True))
            total_vectors += n_vectors
    
    # Set population size equal to or slightly larger than number of reference vectors
    # This ensures each reference vector can have at least one associated solution
    pop_size = total_vectors
    
    # Ensure population size is always an even number
    # If odd, round up to the next even number
    if pop_size % 2 != 0:
        pop_size += 1
    
    # Choose generation count based on problem family
    if problem_name.startswith("dtlz"):
        n_gen = dtlz_gen_defaults.get(n_obj, 300)
    elif problem_name.startswith("wfg"):
        n_gen = wfg_gen_defaults.get(n_obj, 500)
    else:
        raise ValueError(f"Unknown problem family: {problem_name}")

    return pop_size, n_gen, pymoo_layers


def get_ideal_nadir(problem_name: str, n_obj: int):
    """
    Returns the ideal and nadir vectors for a DTLZ or WFG problem,
    along with symbols for each objective.

    Parameters
    ----------
    problem_name : str
        Name of the problem, e.g., "dtlz1", "dtlz2", "wfg3", etc.
    n_obj : int
        Number of objectives.

    Returns
    -------
    symbols : list[str]
        Objective symbols, e.g., ["f_1", "f_2", ...].
    ideal_dict : dict
        Mapping from objective symbol to ideal value.
    nadir_dict : dict
        Mapping from objective symbol to nadir value.
    """
    import numpy as np

    symbols = [f"f_{i+1}" for i in range(n_obj)]

    problem_name = problem_name.lower()

    if problem_name.startswith("dtlz"):
        if problem_name == "dtlz1":
            ideal = np.zeros(n_obj)
            nadir = np.full(n_obj, 0.5)
        elif problem_name in ["dtlz2", "dtlz3", "dtlz4", "dtlz5", "dtlz6"]:
            ideal = np.zeros(n_obj)
            nadir = np.ones(n_obj)
        elif problem_name == "dtlz7":
            ideal = np.zeros(n_obj)
            ideal[-1] = 1.0
            nadir = np.ones(n_obj)
            nadir[-1] = 2.0
        else:
            raise ValueError(f"Unknown DTLZ problem: {problem_name}")

    elif problem_name.startswith("wfg"):
        ideal = np.zeros(n_obj)
        # Nadir: 2 * objective index
        nadir = np.array([2*(i+1) for i in range(n_obj)])
    else:
        raise ValueError(f"Unknown problem family: {problem_name}")

    ideal_dict = dict(zip(symbols, ideal))
    nadir_dict = dict(zip(symbols, nadir))

    return ideal_dict, nadir_dict

def create_problem(name: str, n_obj: int):
    """Create a Problem instance from the FastAPI server."""
    family = "dtlz" if name.startswith("dtlz") else "wfg"
    n_var = get_dtlz_default_nvar(name, n_obj) if family == "dtlz" else get_wfg_default_nvar(name, n_obj)
    params = PymooParameters(name=name, n_var=n_var, n_obj=n_obj)
    problem = server_problem(params)
    ideal, nadir = get_ideal_nadir(name, n_obj)
    problem = problem.update_ideal_and_nadir(new_ideal=ideal, new_nadir=nadir)
    
    print(f"✅ Created {name.upper()} with {n_var} vars and {n_obj} objectives.")
    return problem

## 5. Initialize ADM and Interactive Solvers

Define ADM parameters and initialize ADM

In [4]:
IT_LEARNING = 4
IT_DECISION = 3

Run ADM-based interactive optimization for one problem.

In [5]:
def run_adm_optimization(problem: Problem) -> pd.DataFrame:
    n_obj = len(problem.objectives)
    symbols = [f"{x.symbol}" for x in problem.objectives]
    reference_points = []
    pop_size, n_gen, pymoo_layers = get_default_ea_parameters(problem.name, n_obj)

    # Initialize ADM
    adm = ADMAfsar(
        problem=problem,
        it_learning_phase=IT_LEARNING,
        it_decision_phase=IT_DECISION,
        reference_vector_options=ReferenceVectorOptions(
            creation_type="multi_layer",
            pymoo_layers=pymoo_layers
        ),
        true_ideal=problem.get_ideal_point(),
        true_nadir=problem.get_nadir_point(),
    )

    # Initial solvers
    solver_nsga3, publisher_nsga3 = nsga3(
        problem=problem,
        reference_vector_options=ReferenceVectorOptions(
            reference_point=dict(zip(symbols, adm.preference)),
            creation_type="multi_layer",
            pymoo_layers=pymoo_layers
        )
    )
    solver_rvea, publisher_rvea = rvea(
        problem=problem,
        reference_vector_options=ReferenceVectorOptions(
            reference_point=dict(zip(symbols, adm.preference)),
            creation_type="multi_layer",
            pymoo_layers=pymoo_layers
        )
    )
    archive_nsga3 = NonDominatedArchive(problem=problem, publisher=publisher_nsga3)
    archive_rvea = NonDominatedArchive(problem=problem, publisher=publisher_rvea)
    publisher_nsga3.auto_subscribe(archive_nsga3)
    publisher_rvea.auto_subscribe(archive_rvea)

    results_nsga3 = solver_nsga3()
    results_rvea = solver_rvea()

    iteration = 0
    while adm.has_next():
        iteration += 1
        front_rvea = results_rvea.optimal_outputs.select([symbols[i] for i in range(n_obj)]).to_numpy()
        front_nsga3 = results_nsga3.optimal_outputs.select([symbols[i] for i in range(n_obj)]).to_numpy()

        #print(front_nsga3)
        # Update ADM preference
        adm.get_next_preference(front_rvea, front_nsga3)
       
        # Determine phase
        phase = "L" if iteration <= IT_LEARNING else "D"

        # Store reference point with phase
        reference_points.append(list(adm.preference) + [phase])

        # Rebuild solvers with new reference point
        solver_nsga3, _ = nsga3(
            problem=problem,
            reference_vector_options=ReferenceVectorOptions(
                reference_point=dict(zip(symbols, adm.preference)),
                creation_type="multi_layer",
                pymoo_layers=pymoo_layers
            ),
        )
        solver_rvea, _ = rvea(
            problem=problem,
            reference_vector_options=ReferenceVectorOptions(
                reference_point=dict(zip(symbols, adm.preference)),
                creation_type="multi_layer",
                pymoo_layers=pymoo_layers
            ),
        )

        results_nsga3 = solver_nsga3()
        results_rvea = solver_rvea()

        phase = "Learning" if iteration <= IT_LEARNING else "Decision"

    # Convert to DataFrame
    columns = [f"f_{i+1}" for i in range(n_obj)] + ["Phase"]
    df_ref = pd.DataFrame(reference_points, columns=columns)
    return df_ref

# Example: Create a DTLZ2 problem with 5 objectives
problem = create_problem("dtlz2", 5)

reference_points = run_adm_optimization(problem)
print("\nGenerated Reference Points:")
print(reference_points)
    

✅ Created DTLZ2 with 14 vars and 5 objectives.


C:\Users\Giomara\Documents\Projects\GECCO_journal\DESDEO\desdeo\emo\methods\EAs.py:78: UserWarning: Adaptation frequency was set to 0. Setting it to 100 for RVEA selector. Set it to 0 only if you provide preference information.
  selector = RVEASelector(
C:\Users\Giomara\Documents\Projects\GECCO_journal\DESDEO\desdeo\emo\methods\EAs.py:78: UserWarning: Adaptation frequency was set to 0. Setting it to 100 for RVEA selector. Set it to 0 only if you provide preference information.
  selector = RVEASelector(
C:\Users\Giomara\Documents\Projects\GECCO_journal\DESDEO\desdeo\emo\methods\EAs.py:78: UserWarning: Adaptation frequency was set to 0. Setting it to 100 for RVEA selector. Set it to 0 only if you provide preference information.
  selector = RVEASelector(



Generated Reference Points:
            f_1       f_2       f_3       f_4       f_5 Phase
0  3.036915e-17  0.000000  0.000000  1.003707  0.000000     L
1  1.384737e-34  0.000000  0.000000  0.727711  0.727711     L
2  1.384737e-34  0.537761  0.000000  0.268880  0.806641     L
3  1.384737e-34  0.000000  0.541024  0.270512  0.811535     L
4  1.384737e-34  0.534997  0.000000  0.267499  0.802496     D
5  1.384737e-34  0.534997  0.000000  0.267499  0.802496     D
6  1.384737e-34  0.534997  0.000000  0.267499  0.802496     D


In [6]:
# # 6. Run All Experiments

dtlz_problems = [f"dtlz{i}" for i in range(1, 8)]
#wfg_problems = [f"wfg{i}" for i in range(1, 10)]
objective_counts = [3,5]

total = len(dtlz_problems ) * len(objective_counts)
counter = 0

start_time = time.time()
for name in dtlz_problems:
    for n_obj in objective_counts:
        counter += 1
        print(f"\n--- [{counter}/{total}] Running {name.upper()} with {n_obj} objectives ---")
        problem = create_problem(name, n_obj)
        reference_points = run_adm_optimization(problem)
        out_path = os.path.join(OUTPUT_DIR, f"{name}_{n_obj}obj_refpoints.csv")
        reference_points.to_csv(out_path, index=False)
        print(f"💾 Saved {len(reference_points)} reference points → {out_path}")

elapsed = (time.time() - start_time) / 60
print(f"\n✅ Completed all experiments in {elapsed:.2f} minutes.")



--- [1/14] Running DTLZ1 with 3 objectives ---
✅ Created DTLZ1 with 7 vars and 3 objectives.


C:\Users\Giomara\Documents\Projects\GECCO_journal\DESDEO\desdeo\emo\methods\EAs.py:78: UserWarning: Adaptation frequency was set to 0. Setting it to 100 for RVEA selector. Set it to 0 only if you provide preference information.
  selector = RVEASelector(


💾 Saved 7 reference points → results\dtlz1_3obj_refpoints.csv

--- [2/14] Running DTLZ1 with 5 objectives ---
✅ Created DTLZ1 with 9 vars and 5 objectives.


C:\Users\Giomara\Documents\Projects\GECCO_journal\DESDEO\desdeo\emo\operators\selection.py:1230: RuntimeWarning: divide by zero encountered in divide
  intercepts = 1 / plane


💾 Saved 7 reference points → results\dtlz1_5obj_refpoints.csv

--- [3/14] Running DTLZ2 with 3 objectives ---
✅ Created DTLZ2 with 12 vars and 3 objectives.
💾 Saved 7 reference points → results\dtlz2_3obj_refpoints.csv

--- [4/14] Running DTLZ2 with 5 objectives ---
✅ Created DTLZ2 with 14 vars and 5 objectives.
💾 Saved 7 reference points → results\dtlz2_3obj_refpoints.csv

--- [4/14] Running DTLZ2 with 5 objectives ---
✅ Created DTLZ2 with 14 vars and 5 objectives.
💾 Saved 7 reference points → results\dtlz2_5obj_refpoints.csv

--- [5/14] Running DTLZ3 with 3 objectives ---
✅ Created DTLZ3 with 12 vars and 3 objectives.
💾 Saved 7 reference points → results\dtlz2_5obj_refpoints.csv

--- [5/14] Running DTLZ3 with 3 objectives ---
✅ Created DTLZ3 with 12 vars and 3 objectives.
💾 Saved 7 reference points → results\dtlz3_3obj_refpoints.csv

--- [6/14] Running DTLZ3 with 5 objectives ---
✅ Created DTLZ3 with 14 vars and 5 objectives.
💾 Saved 7 reference points → results\dtlz3_3obj_refpoints.

C:\Users\Giomara\Documents\Projects\GECCO_journal\DESDEO\desdeo\emo\methods\EAs.py:78: UserWarning: Adaptation frequency was set to 0. Setting it to 100 for RVEA selector. Set it to 0 only if you provide preference information.
  selector = RVEASelector(


💾 Saved 7 reference points → results\dtlz4_3obj_refpoints.csv

--- [8/14] Running DTLZ4 with 5 objectives ---
✅ Created DTLZ4 with 14 vars and 5 objectives.
💾 Saved 7 reference points → results\dtlz4_5obj_refpoints.csv

--- [9/14] Running DTLZ5 with 3 objectives ---
✅ Created DTLZ5 with 12 vars and 3 objectives.
💾 Saved 7 reference points → results\dtlz4_5obj_refpoints.csv

--- [9/14] Running DTLZ5 with 3 objectives ---
✅ Created DTLZ5 with 12 vars and 3 objectives.
💾 Saved 7 reference points → results\dtlz5_3obj_refpoints.csv

--- [10/14] Running DTLZ5 with 5 objectives ---
✅ Created DTLZ5 with 14 vars and 5 objectives.
💾 Saved 7 reference points → results\dtlz5_3obj_refpoints.csv

--- [10/14] Running DTLZ5 with 5 objectives ---
✅ Created DTLZ5 with 14 vars and 5 objectives.
💾 Saved 7 reference points → results\dtlz5_5obj_refpoints.csv

--- [11/14] Running DTLZ6 with 3 objectives ---
✅ Created DTLZ6 with 12 vars and 3 objectives.
💾 Saved 7 reference points → results\dtlz5_5obj_refpoin

C:\Users\Giomara\Documents\Projects\GECCO_journal\DESDEO\desdeo\emo\methods\EAs.py:78: UserWarning: Adaptation frequency was set to 0. Setting it to 100 for RVEA selector. Set it to 0 only if you provide preference information.
  selector = RVEASelector(


💾 Saved 7 reference points → results\dtlz7_3obj_refpoints.csv

--- [14/14] Running DTLZ7 with 5 objectives ---
✅ Created DTLZ7 with 24 vars and 5 objectives.
💾 Saved 7 reference points → results\dtlz7_5obj_refpoints.csv

✅ Completed all experiments in 29.39 minutes.
💾 Saved 7 reference points → results\dtlz7_5obj_refpoints.csv

✅ Completed all experiments in 29.39 minutes.
